# 选修E1 · Day 3：多Agent系统设计 · 上机练习（v5.0）

> **真实库**：LangGraph（多Agent协作图）+ networkx（拓扑分析）
> **核心任务**：4个营销Agent（researcher/strategist/writer/reviewer）协作完成营销策略产出
> **拓扑对比**：supervisor中心化 vs team去中心化
> **营销映射**：透肌精华竞品分析（雅诗兰黛）+ 策略 + 文案 + 审核，多Agent涌现团队决策

本笔记本构建一个完整的多Agent营销协作系统。你将：
1. 定义Agent间通信协议（AgentMessage）和共享状态（MultiAgentState）
2. 实现4个营销Agent节点（researcher/strategist/writer/reviewer）
3. 用LangGraph构建supervisor中心化拓扑和team去中心化拓扑
4. 用networkx分析两种拓扑的中心性/连通性/瓶颈
5. 运行双拓扑系统，分析涌现行为，映射天道推演

> 天道推演视角：多Agent系统是"可计算沙盘"--supervisor模拟决策者，4个Agent模拟利益相关方，networkx量化推演拓扑质量


In [ ]:
# === 导入真实库 ===
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from langgraph.graph import StateGraph, END, START
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from enum import Enum
import operator
import networkx as nx
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# === 真实营销数据（基于护肤品电商场景，复用Day 1/2）===
PRODUCT_DB = {
    "透肌精华": "透肌焕亮精华液，299元，主打美白焕亮，含烟酰胺3%+维C衍生物，目标用户25-35岁都市白领。",
    "玻尿酸面霜": "玻尿酸保湿面霜，159元，主打深层补水，含双重玻尿酸，目标用户18-30岁女性。",
}
COMPETITOR_DB = {
    "雅诗兰黛": "雅诗兰黛小棕瓶精华，760元/30ml，市场占有率18%，优势：品牌力强、渠道完善；劣势：价格高、年轻化不足。",
    "兰蔻": "兰蔻小黑瓶精华，780元/30ml，市场占有率15%，优势：科技感强、专柜体验；劣势：下沉市场覆盖弱。",
}

# === 统一营销任务（所有Agent协作完成同一个任务）===
MARKETING_TASK = "为透肌精华制定营销策略，竞品分析雅诗兰黛，产出合规文案"

# === 离线模拟LLM（无需API Key，预编排Agent响应）===
class StubChatModel(BaseChatModel):
    """离线模拟LLM，预编排Agent响应序列，保证无API Key可运行。
    替换为ChatOpenAI/ChatAnthropic即可使用真实LLM驱动多Agent涌现。"""
    responses: list = []
    call_index: int = 0

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        idx = self.call_index
        self.call_index += 1
        if idx < len(self.responses):
            resp = self.responses[idx]
        else:
            resp = AIMessage(content="Agent任务完成。")
        return ChatResult(generations=[ChatGeneration(message=resp)])

    @property
    def _llm_type(self):
        return "stub"

print("真实库导入成功")
print(f"  LangGraph: StateGraph + add_conditional_edges (多Agent协作图)")
print(f"  networkx: {nx.__version__} (Agent通信拓扑分析)")
print(f"  产品库: {list(PRODUCT_DB.keys())}")
print(f"  竞品库: {list(COMPETITOR_DB.keys())}")
print(f"  统一营销任务: {MARKETING_TASK}")
print(f"  StubChatModel: 离线模式（无API Key可运行）")

---
## TODO1：定义Agent间通信协议和共享状态

多Agent系统的核心是**通信**和**状态共享**。需要定义：

1. **MessageType枚举**：任务分配/结果汇报/信息共享/请求帮助/反馈/投票
2. **AgentMessage（pydantic BaseModel）**：sender/receiver/message_type/content/metadata/reply_to
3. **MultiAgentState（TypedDict）**：所有Agent共享的全局状态，包含task/messages/research_data/strategy/content/review_result/current_agent/revision_count/approved

这是A2A协议（Agent间互操作）的简化实现。真实A2A协议还包含Agent Card发现和任务状态查询，本Day聚焦消息格式层。

> 天道推演对应：AgentMessage是"因果链追踪"的载体，MultiAgentState是"沙盘"的当前状态


In [ ]:
# TODO1: 定义Agent间通信协议和共享状态
# 提示:
#   1. class MessageType(Enum): TASK_ASSIGNMENT / RESULT_REPORT / INFO_SHARING / HELP_REQUEST / FEEDBACK / VOTE
#   2. class AgentMessage(BaseModel): sender/receiver/message_type/content/metadata/reply_to
#   3. class MultiAgentState(TypedDict): task/messages/research_data/strategy/content/review_result/current_agent/revision_count/approved
#      messages字段用 Annotated[list, operator.add] 实现累积传递

# TODO: 你的代码
raise NotImplementedError

---
## TODO2：实现4个营销Agent节点函数

每个Agent是一个节点函数：接收State，返回State更新。4个Agent各有职责：

| Agent | 职责 | 读State | 写State |
|-------|------|---------|---------|
| researcher | 市场调研 | task | research_data + messages |
| strategist | 策略制定 | research_data | strategy + messages |
| writer | 文案生成 | strategy | content + messages |
| reviewer | 合规审核 | content | review_result + approved + messages |

每个Agent执行后向State追加AgentMessage（message_type=RESULT_REPORT），实现通信可追溯。

> 天道推演对应：researcher=局势感知，strategist=沙盘模拟，writer=最优路径推荐，reviewer=反馈学习


In [ ]:
# TODO2: 实现4个营销Agent节点函数
# 提示:
#   1. researcher_agent(state): 查询PRODUCT_DB和COMPETITOR_DB，返回 research_data 和 messages 更新
#   2. strategist_agent(state): 基于research_data制定差异化策略，返回 strategy 和 messages
#   3. writer_agent(state): 基于strategy写营销文案，返回 content 和 messages
#   4. reviewer_agent(state): 审核content合规性，返回 review_result/approved 和 messages
#   每个Agent返回的messages是 [AgentMessage(...)] 列表，通过operator.add累积

# TODO: 你的代码
raise NotImplementedError

---
## TODO3：用LangGraph构建supervisor中心化拓扑

**supervisor拓扑**（Hub-and-Spoke）：一个supervisor节点负责任务分配和结果汇总，其他Agent各自执行分配的子任务。

```
         supervisor
        /    |    |    \
  researcher strategist writer reviewer
```

用LangGraph实现：
1. `supervisor_node`：根据state的`current_agent`决定下一步路由（researcher->strategist->writer->reviewer->END）
2. 4个Agent节点（TODO2已实现）
3. `add_conditional_edges`：supervisor到各Agent的条件路由
4. Agent执行后返回supervisor（形成星型拓扑）

> 天道推演对应：supervisor=因果链追踪（集中调度），星型拓扑可控但supervisor是单点故障


In [ ]:
# TODO3: 用LangGraph构建supervisor中心化拓扑
# 提示:
#   1. supervisor_node(state): 根据 current_agent 决定路由
#      researcher->strategist->writer->reviewer->END
#   2. route_from_supervisor(state): 返回下一Agent名称或 "END"
#   3. StateGraph(MultiAgentState): add_node(supervisor/researcher/strategist/writer/reviewer)
#   4. add_edge(START, "supervisor")
#   5. add_conditional_edges("supervisor", route_from_supervisor, {...})
#   6. 各Agent节点 add_edge 回 "supervisor"
#   7. compile() 得到 supervisor_app

# TODO: 你的代码
raise NotImplementedError

---
## TODO4：用LangGraph构建team去中心化拓扑

**team拓扑**（去中心化）：无中心supervisor，Agent间直接传递消息。每个Agent完成后直接路由到下一个Agent，形成流水线或网状结构。

```
  researcher -> strategist -> writer -> reviewer -> END
```

用LangGraph实现：
1. 无supervisor节点，Agent间直接`add_edge`连接
2. researcher->strategist->writer->reviewer->END（流水线式直接传递）
3. 消息通过State的`messages`字段累积传递（operator.add reducer）

对比supervisor拓扑：去中心化减少了一跳通信，但失去了集中控制能力。

> 天道推演对应：team=自由协作（无中心决策者），需依赖Agent间协议自组织


In [ ]:
# TODO4: 用LangGraph构建team去中心化拓扑
# 提示:
#   1. 无supervisor节点，Agent间直接连接
#   2. add_edge(START, "researcher")
#   3. add_edge("researcher", "strategist")
#   4. add_edge("strategist", "writer")
#   5. add_edge("writer", "reviewer")
#   6. add_edge("reviewer", END)
#   7. compile() 得到 team_app

# TODO: 你的代码
raise NotImplementedError

---
## TODO5：用networkx分析两种Agent通信拓扑

将多Agent系统的通信关系建模为有向图：
- 节点 = Agent（supervisor/researcher/strategist/writer/reviewer）
- 边 = 消息流（谁向谁发消息）

用networkx分析：
1. 构建supervisor拓扑图和team拓扑图
2. `nx.degree_centrality()`：度中心性（谁是通信枢纽）
3. `nx.betweenness_centrality()`：介数中心性（谁是信息瓶颈）
4. `nx.is_strongly_connected()`：强连通性（消息能否到达所有Agent）
5. 识别瓶颈Agent和单点故障风险

> 天道推演对应：networkx指标量化"因果链追踪"--哪个Agent是关键因果节点？


In [ ]:
# TODO5: 用networkx分析两种Agent通信拓扑
# 提示:
#   1. 构建supervisor拓扑图: nx.DiGraph(), 节点=Agent, 边=消息流
#      supervisor<->researcher, supervisor<->strategist, ... (双向)
#   2. 构建team拓扑图: researcher->strategist->writer->reviewer (单向流水线)
#   3. nx.degree_centrality(G): 计算两种拓扑的度中心性
#   4. nx.betweenness_centrality(G): 计算介数中心性
#   5. nx.is_strongly_connected(G): 判断强连通性
#   6. 打印对比表

# TODO: 你的代码
raise NotImplementedError

---
## TODO6：运行双拓扑系统 + 涌现行为分析 + 天道推演映射

运行supervisor_app和team_app，对比：
1. **执行轨迹**：两种拓扑的Agent调用序列和消息流
2. **涌现指标**：用networkx指标量化涌现质量（通信效率/决策质量/鲁棒性）
3. **天道推演映射**：将多Agent仿真映射到天道推演六能力

> 天道推演核心：多Agent系统是"可计算沙盘"--通过不同拓扑的涌现行为对比，推演哪种拓扑在营销场景下产出更优决策。这把项目CLAUDE.md的「天道推演系统」从思维框架升级为可计算的多Agent仿真工具。


In [ ]:
# TODO6: 运行双拓扑系统 + 涌现行为分析 + 天道推演映射
# 提示:
#   1. 初始化state: {task, messages:[], research_data:"", strategy:"", content:"",
#      review_result:"", current_agent:"init", revision_count:0, approved:False}
#   2. supervisor_app.invoke(state): 运行supervisor拓扑
#   3. team_app.invoke(state): 运行team拓扑
#   4. 打印两种拓扑的执行结果和消息流
#   5. 分析涌现行为: 通信效率/决策质量/鲁棒性
#   6. 映射天道推演六能力

# TODO: 你的代码
raise NotImplementedError